In [1]:
import torch
import transformers
import ewok_eval
from transformers import AutoModelForCausalLM, AutoTokenizer

# from ewok_eval import per_token_conditional_log_likelihood,batch_per_token_conditional_log_likelihood, per_token_log_likelihood

In [2]:
#we load the babylm gpt-2 model
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "BabyLM-community/babylm-baseline-10m-gpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = model.to(device)

In [3]:
import torch

def per_token_log_likelihood(model, tokenizer, input_texts, device="cuda"):
    # Tokenize the batch with padding
    # return_tensors="pt" creates a rectangular tensor (Batch, Max_Seq_Len)
    inputs = tokenizer(input_texts, add_special_tokens=False, return_tensors="pt", padding=True)
    input_ids = inputs.input_ids.to(device)
    attn_mask = inputs.attention_mask.to(device)
    
    # Get batch size
    batch_size = input_ids.shape[0]

    # Prepend BOS token (Batch-wise)
    bos_token_id = tokenizer.bos_token_id
    bos_tensor = torch.full((batch_size, 1), bos_token_id, device=device)
    input_ids = torch.cat([bos_tensor, input_ids], dim=1)

    # Prepend Attention Mask (Batch-wise)
    ones_tensor = torch.ones((batch_size, 1), device=device)
    attn_mask = torch.cat([ones_tensor, attn_mask], dim=1)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attn_mask)
        logits = outputs['logits']

    # Remove last logit (shift left)
    logits = logits[:, :-1, :]
    log_probs = torch.log_softmax(logits, dim=-1)

    # Remove first input_id (BOS) for aligning labels
    input_ids = input_ids[:, 1:]

    # Gather log probs at the specific input indices
    token_logprobs = log_probs.gather(dim=-1, index=input_ids.unsqueeze(-1)).squeeze(-1)

    # Note: token_logprobs now contains values for PADDING tokens at the end too.
    # We return the whole thing and let the caller slice it.
    return token_logprobs, inputs.attention_mask

def per_token_conditional_log_likelihood(model, tokenizer, contexts, targets, device="cuda"):
    # 1. Prepare batch of texts exactly as you did
    texts = [c + " " + t for c, t in zip(contexts, targets)]
    
    # 2. Calculate context lengths for each item in the batch
    # We do this in a loop because they can be different lengths
    context_token_lengths = [len(tokenizer.encode(c, add_special_tokens=False)) for c in contexts]

    # 3. Run the model on the batch (High Performance)
    batch_log_probs, batch_attn_masks = per_token_log_likelihood(model, tokenizer, texts, device)

    # 4. Slice the results row-by-row to match your original logic
    results = []
    for i in range(len(contexts)):
        # Get the start index for this specific row
        start_idx = context_token_lengths[i]
        
        # Get the total length of valid tokens (excluding padding)
        # sum() gives the count of real tokens (non-padded)
        valid_length = batch_attn_masks[i].sum().item()
        
        # Slice: from context_end up to the actual end of the sentence (ignoring padding)
        # We use valid_length because batching adds padding zeros at the end which we don't want
        row_result = batch_log_probs[i, start_idx:valid_length]
        
        results.append(row_result)

    return results
def per_token_conditional_log_likelihood(model, tokenizer, contexts, targets, device="cuda", batch_size=8):
    all_results = []
    
    # Process data in chunks of batch_size
    for i in range(0, len(contexts), batch_size):
        # 1. Slice the current batch
        batch_contexts = contexts[i : i + batch_size]
        batch_targets = targets[i : i + batch_size]
        
        # 2. Prepare batch of texts (Context + " " + Target)
        batch_texts = [c + " " + t for c, t in zip(batch_contexts, batch_targets)]
        
        # 3. Calculate context lengths for this specific batch
        #    (We need this to know where to start slicing the probabilities)
        batch_context_lengths = [len(tokenizer.encode(c, add_special_tokens=False)) for c in batch_contexts]

        # 4. Run the model on the current batch
        #    per_token_log_likelihood handles padding and tokenization internally
        batch_log_probs, batch_attn_masks = per_token_log_likelihood(model, tokenizer, batch_texts, device)

        # 5. Process results for this batch
        for j in range(len(batch_contexts)):
            # Get the start index for this specific row
            start_idx = batch_context_lengths[j]
            
            # Get the total length of valid tokens (excluding padding)
            valid_length = batch_attn_masks[j].sum().item()
            
            # Slice: from context_end up to the actual end of the sentence
            # We assume batch_log_probs[j] corresponds to batch_texts[j]
            row_result = batch_log_probs[j, start_idx:valid_length]
            
            all_results.append(row_result)

    return all_results
contexts = ["Jesse applied pressure to the balloon. It remained intact."]
targets = ["The balloon is fragile."]

# Returns a list of tensors
results = per_token_conditional_log_likelihood(model, tokenizer, contexts, targets)

# Check the sum for the first item
print(results[0].sum())

tensor(-19.9060, device='cuda:0')


In [4]:
# results[0]
# temp = tokenizer.encode(contexts[0]+ " " + targets[0], add_special_tokens=False)
# tokenizer.decode(temp[-6:])


In [5]:
# we define a list of contexts and targets
contexts = [
    "The cat sat on the",
    "In a galaxy far, far",
    "Once upon a time in a",
]
targets = [
    " mat.",
    " away.",
    " land. Hey",
]
# we compute the per-token conditional log-likelihoods
# results = batch_per_token_conditional_log_likelihood(model, tokenizer, contexts, targets)
results
import pandas as pd
from pathlib import Path

SRC = Path("/home/jorge/tokenPred/babylm_10m/test_eval/evaluation-pipeline-2025/evaluation_data/fast_eval/ewok_fast")          # adjust path if needed

ewok_df = pd.concat(
    [
        pd.read_json(fp, lines=True, encoding="utf-8")
        #   .assign(file=fp.name, category=fp.stem)  # metadata
        for fp in SRC.glob("*.jsonl")              # use rglob("*.jsonl") if nested
    ],
    ignore_index=True,
    sort=False,    # keep union of columns
)

# optional niceties
ewok_df = ewok_df.convert_dtypes()


In [6]:
ewok_df

,Domain,ConceptA,ConceptB,ContextType,ContextDiff,TargetDiff,Context1,Context2,Target1,Target2
0,material-properties,fragile,sturdy,direct,antonym,concept swap,Jesse applied pressure to the balloon. It rema...,Jesse applied pressure to the balloon. It broke.,The balloon is sturdy.,The balloon is fragile.
1,material-properties,transparent,opaque,direct,negation,concept swap,Chao cannot see through the wheel.,Chao can see through the wheel.,The wheel is opaque.,The wheel is transparent.
2,material-properties,soft,hard,indirect,material,concept swap,The bin is made of jelly.,The bin is made of glass.,The bin is soft.,The bin is hard.
3,material-properties,soft,hard,indirect,material,concept swap,The balloon is made of terracotta.,The balloon is made of fabric.,The surface of the balloon is hard.,The surface of the balloon is soft.
4,material-properties,heavy,light,direct,antonym,concept swap,The balloon weighs a lot.,The balloon weighs little.,The balloon is heavy.,The balloon is light.
...,...,...,...,...,...,...,...,...,...,...
1095,material-dynamics,drape,stir,indirect,material,concept swap,Ali sees the propane.,Ali sees the twill.,Ali stirs it.,Ali drapes it.
1096,material-dynamics,drape,stir,direct,material,concept swap,Chao sees something that is liquid.,Chao sees something that is fabric.,Chao stirs it.,Chao drapes it.
1097,material-dynamics,squeeze,stir,indirect,material,concept swap,Mohammed sees the nitrogen.,Mohammed sees the play-doh.,Mohammed stirs it.,Mohammed squeezes it.
1098,material-dynamics,break,drip,indirect,material,concept swap,Chao sees the water.,Chao sees the feather.,It drips.,It breaks.


In [ ]:
def evaluate(model,tokenizer, ewok_df):
    domains = ewok_df['Domain'].unique()
    # domain = 'spatial-relations'
    domain_scores_official = {}
    domain_scores_full = {}
    for domain in domains:
        print(domain)
        bs = 2
        df = ewok_df[ewok_df['Domain'] == domain]
        # print(len(df))
        #we apply the per_token_conditional_log_likelihood function to the ewok dataset for each row
        context1 = df["Context1"].tolist()
        target1 = df["Target1"].tolist()
        context2 = df["Context2"].tolist()
        target2 = df["Target2"].tolist()
        #we check to make sure the model assigns higher likelihood P(T_1|C_1) > P(T_1|C_2) and P(T_2|C_2) > P(T_2|C_1)
        results_1_1 = per_token_conditional_log_likelihood(model, tokenizer, context1, target1, batch_size=bs)
        results_1_2 = per_token_conditional_log_likelihood(model, tokenizer, context1, target2, batch_size=bs)
        results_2_2 = per_token_conditional_log_likelihood(model, tokenizer, context2, target2, batch_size=bs)
        results_2_1 = per_token_conditional_log_likelihood(model, tokenizer, context2, target1, batch_size=bs)
        import numpy as np
        correct_1 = []
        for r1, r2 in zip(results_1_1, results_1_2):
            sum_r1 = r1.sum().item()
            sum_r2 = r2.sum().item()
            correct_1.append(sum_r1 > sum_r2)
        correct_2 = []
        for r1, r2 in zip(results_2_2, results_2_1):
            sum_r1 = r1.sum().item()
            sum_r2 = r2.sum().item()
            correct_2.append(sum_r1 > sum_r2)
        accuracy_1 = np.mean(correct_1)
        accuracy_2 = np.mean(correct_2)
        print(f"Accuracy for Target1: {accuracy_1*100:.2f}%")
        print(f"Accuracy for Target2: {accuracy_2*100:.2f}%")
        domain_scores_full[domain] = (accuracy_1.item(), accuracy_2.item())
        # we store only the accuracy for Target1 as official score,
        #as that matches the original ewok evaluation protocol
        domain_scores_official[domain] = accuracy_1.item() 
    return domain_scores_official, domain_scores_full
evaluate(model,tokenizer, ewok_df)
    # print(f"Overall Accuracy: {(accuracy_1 + accuracy_2)/2*100:.2f}%")

material-properties
Accuracy for Target1: 45.00%
Accuracy for Target2: 51.00%
social-interactions
Accuracy for Target1: 54.00%
Accuracy for Target2: 47.00%
social-relations
Accuracy for Target1: 52.00%
Accuracy for Target2: 50.00%
spatial-relations
Accuracy for Target1: 43.00%
Accuracy for Target2: 50.00%
quantitative-properties
Accuracy for Target1: 53.00%
Accuracy for Target2: 51.00%
physical-dynamics
Accuracy for Target1: 48.00%
Accuracy for Target2: 52.00%
agent-properties
Accuracy for Target1: 55.00%
Accuracy for Target2: 50.00%
social-properties
Accuracy for Target1: 57.00%
Accuracy for Target2: 47.00%
physical-relations
Accuracy for Target1: 50.00%
Accuracy for Target2: 49.00%
physical-interactions
Accuracy for Target1: 48.00%
Accuracy for Target2: 53.00%
material-dynamics
Accuracy for Target1: 45.00%
Accuracy for Target2: 53.00%


({'material-properties': 0.45,
  'social-interactions': 0.54,
  'social-relations': 0.52,
  'spatial-relations': 0.43,
  'quantitative-properties': 0.53,
  'physical-dynamics': 0.48,
  'agent-properties': 0.55,
  'social-properties': 0.57,
  'physical-relations': 0.5,
  'physical-interactions': 0.48,
  'material-dynamics': 0.45},
 {'material-properties': (np.float64(0.45), np.float64(0.51)),
  'social-interactions': (np.float64(0.54), np.float64(0.47)),
  'social-relations': (np.float64(0.52), np.float64(0.5)),
  'spatial-relations': (np.float64(0.43), np.float64(0.5)),
  'quantitative-properties': (np.float64(0.53), np.float64(0.51)),
  'physical-dynamics': (np.float64(0.48), np.float64(0.52)),
  'agent-properties': (np.float64(0.55), np.float64(0.5)),
  'social-properties': (np.float64(0.57), np.float64(0.47)),
  'physical-relations': (np.float64(0.5), np.float64(0.49)),
  'physical-interactions': (np.float64(0.48), np.float64(0.53)),
  'material-dynamics': (np.float64(0.45), np.floa

In [37]:
ewok_df['Target2'].tolist()


['The balloon is fragile.',
 'The wheel is transparent.',
 'The bin is hard.',
 'The surface of the balloon is soft.',
 'The balloon is light.',
 'The balloon is fragile.',
 'The surface of the balloon is warm.',
 'The surface of the bin is soft.',
 'The wheel is not bouncy.',
 'The balloon is bouncy.',
 'The bin is not bouncy.',
 'The volleyball is transparent.',
 'The football is not bouncy.',
 'The wheel is opaque.',
 'The surface of the volleyball is soft.',
 'The wheel is bouncy.',
 'The football is bouncy.',
 'The volleyball is sturdy.',
 'The volleyball is soft.',
 'The surface of the football is soft.',
 'The surface of the volleyball is hard.',
 'The football is sturdy.',
 'The wheel is not bouncy.',
 'The bin is heavy.',
 'The volleyball is not bouncy.',
 'The wheel is fragile.',
 'The surface of the volleyball is smooth.',
 'The volleyball is fragile.',
 'The surface of the balloon is rough.',
 'The surface of the bin is rough.',
 'The bin is not bouncy.',
 'The surface of t

In [38]:
import torch
import torch.nn.functional as F

def per_token_log_likelihood(model, tokenizer, input_texts, device="cuda", batch_size=8):
    model.eval()
    all_token_logprobs = []
    all_attn_masks = []

    for i in range(0, len(input_texts), batch_size):
        batch_texts = input_texts[i:i + batch_size]

        inputs = tokenizer(batch_texts, add_special_tokens=False, return_tensors="pt", padding=True)
        input_ids = inputs.input_ids.to(device)
        attn_mask = inputs.attention_mask.to(device)

        cur_batch_size = input_ids.shape[0]

        # Prepend BOS token (Batch-wise)
        bos_token_id = tokenizer.bos_token_id
        bos_tensor = torch.full((cur_batch_size, 1), bos_token_id, device=device)
        input_ids = torch.cat([bos_tensor, input_ids], dim=1)

        # Prepend Attention Mask (Batch-wise)
        ones_tensor = torch.ones((cur_batch_size, 1), device=device)
        attn_mask = torch.cat([ones_tensor, attn_mask], dim=1)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attn_mask)
            logits = outputs["logits"]

        # Remove last logit (shift left)
        logits = logits[:, :-1, :]
        log_probs = torch.log_softmax(logits, dim=-1)

        # Remove first input_id (BOS) for aligning labels
        input_ids = input_ids[:, 1:]

        # Gather log probs at the specific input indices
        token_logprobs = log_probs.gather(dim=-1, index=input_ids.unsqueeze(-1)).squeeze(-1)

        all_token_logprobs.append(token_logprobs.cpu())
        all_attn_masks.append(inputs.attention_mask.cpu())

    # Pad to global max length so cat works
    max_len = max(t.shape[1] for t in all_token_logprobs)

    all_token_logprobs = [
        F.pad(t, (0, max_len - t.shape[1]), value=0.0) if t.shape[1] < max_len else t
        for t in all_token_logprobs
    ]
    all_attn_masks = [
        F.pad(m, (0, max_len - m.shape[1]), value=0) if m.shape[1] < max_len else m
        for m in all_attn_masks
    ]

    return torch.cat(all_token_logprobs, dim=0), torch.cat(all_attn_masks, dim=0)


#we are going to compare that per_token_conditional_log_likelihood function with P(T_1,C_1) - P(C_1) > P(T_2,C_1) - P(C_1)
# and P(T_2,C_2) - P(C_2) > P(T_1,C_2) - P(C_2)
results_joint_1_1 = per_token_log_likelihood(model, tokenizer, [c + " " + t for c, t in zip(context1, target1)], batch_size=bs)
results_context_1 = per_token_log_likelihood(model, tokenizer, context1, batch_size=bs)
results_joint_1_2 = per_token_log_likelihood(model, tokenizer, [c + " " + t for c, t in zip(context2, target1)], batch_size=bs)
results_joint_2_2 = per_token_log_likelihood(model, tokenizer, [c + " " + t for c, t in zip(context2, target2)], batch_size=bs)
results_context_2 = per_token_log_likelihood(model, tokenizer, context2, batch_size=bs)
results_joint_2_1 = per_token_log_likelihood(model, tokenizer, [c + " " + t for c, t in zip(context1, target2)], batch_size=bs)
correct_1_alt = []
for i in range(len(context1)):
    joint_1_1 = results_joint_1_1[0][i] * results_joint_1_1[1][i]
    context_1 = results_context_1[0][i] * results_context_1[1][i]
    joint_2_1 = results_joint_2_1[0][i] * results_joint_2_1[1][i]
    sum_r1 = (joint_1_1 ).sum().item() - context_1.sum().item()
    sum_r2 = (joint_2_1 ).sum().item() - context_1.sum().item()
    correct_1_alt.append(sum_r1 > sum_r2)
correct_2_alt = []
for i in range(len(context2)):
    joint_2_2 = results_joint_2_2[0][i] * results_joint_2_2[1][i]
    context_2 = results_context_2[0][i] * results_context_2[1][i]
    joint_1_2 = results_joint_1_2[0][i] * results_joint_1_2[1][i]
    sum_r1 = (joint_2_2 ).sum().item() - context_2.sum().item()
    sum_r2 = (joint_1_2 ).sum().item() - context_2.sum().item()
    correct_2_alt.append(sum_r1 > sum_r2)
accuracy_1_alt = np.mean(correct_1_alt)
accuracy_2_alt = np.mean(correct_2_alt)
print(f"Alternative Accuracy for Target1: {accuracy_1_alt*100:.2f}%")
print(f"Alternative Accuracy for Target2: {accuracy_2_alt*100:.2f}%")

Alternative Accuracy for Target1: 45.00%
Alternative Accuracy for Target2: 51.00%


In [40]:
import torch
import torch.nn.functional as F
import numpy as np

def per_token_log_likelihood(model, tokenizer, input_texts, device="cuda", batch_size=8):
    model.eval()

    # PAD fix (needed for GPT-2 tokenizers)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = tokenizer.pad_token_id

    bos_token_id = tokenizer.bos_token_id
    if bos_token_id is None:
        bos_token_id = tokenizer.eos_token_id

    all_token_logprobs = []
    all_token_masks = []

    for i in range(0, len(input_texts), batch_size):
        batch_texts = input_texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            add_special_tokens=False,
            return_tensors="pt",
            padding=True,
            truncation=True,  # prevents context-too-long crashes
        )
        input_ids = inputs.input_ids.to(device)
        attn_mask = inputs.attention_mask.to(device)

        B = input_ids.shape[0]

        # prepend BOS
        bos = torch.full((B, 1), bos_token_id, device=device, dtype=input_ids.dtype)
        input_ids = torch.cat([bos, input_ids], dim=1)

        ones = torch.ones((B, 1), device=device, dtype=attn_mask.dtype)
        attn_mask = torch.cat([ones, attn_mask], dim=1)

        with torch.no_grad():
            logits = model(input_ids=input_ids, attention_mask=attn_mask).logits  # (B, L, V)

        # predict token t_k using prefix up to t_{k-1}
        log_probs = torch.log_softmax(logits[:, :-1, :], dim=-1)  # (B, L-1, V)
        labels = input_ids[:, 1:]                                # (B, L-1)

        token_logprobs = log_probs.gather(-1, labels.unsqueeze(-1)).squeeze(-1)  # (B, L-1)
        token_mask = attn_mask[:, 1:]  # aligns exactly with token_logprobs

        all_token_logprobs.append(token_logprobs.cpu())
        all_token_masks.append(token_mask.cpu())

    # pad within this call so cat works
    max_len = max(t.shape[1] for t in all_token_logprobs)

    all_token_logprobs = [
        F.pad(t, (0, max_len - t.shape[1]), value=0.0) if t.shape[1] < max_len else t
        for t in all_token_logprobs
    ]
    all_token_masks = [
        F.pad(m, (0, max_len - m.shape[1]), value=0) if m.shape[1] < max_len else m
        for m in all_token_masks
    ]

    return torch.cat(all_token_logprobs, dim=0), torch.cat(all_token_masks, dim=0)


def conditional_scores(model, tokenizer, contexts, targets, bs=8, device="cuda", normalize=True):
    joint_texts = [(c.rstrip() + " " + t.lstrip()) for c, t in zip(contexts, targets)]

    joint_lp, joint_mask = per_token_log_likelihood(model, tokenizer, joint_texts, device=device, batch_size=bs)
    ctx_lp, ctx_mask = per_token_log_likelihood(model, tokenizer, contexts, device=device, batch_size=bs)

    joint_sum = (joint_lp * joint_mask).sum(dim=1)
    ctx_sum = (ctx_lp * ctx_mask).sum(dim=1)

    cond_sum = joint_sum - ctx_sum

    if not normalize:
        return cond_sum

    # length-normalize by target token count
    target_len = (joint_mask.sum(dim=1) - ctx_mask.sum(dim=1)).clamp(min=1)
    cond_avg = cond_sum / target_len
    return cond_avg
s11 = conditional_scores(model, tokenizer, context1, target1, bs=bs, normalize=False)
s12 = conditional_scores(model, tokenizer, context1, target2, bs=bs, normalize=False)
acc1 = (s11 > s12).float().mean().item()

s22 = conditional_scores(model, tokenizer, context2, target2, bs=bs, normalize=False)
s21 = conditional_scores(model, tokenizer, context2, target1, bs=bs, normalize=False)
acc2 = (s22 > s21).float().mean().item()

print(f"Accuracy for Context1 choosing Target1: {acc1*100:.2f}%")
print(f"Accuracy for Context2 choosing Target2: {acc2*100:.2f}%")
print(f"Balanced accuracy: {(acc1+acc2)/2*100:.2f}%")


Accuracy for Context1 choosing Target1: 45.00%
Accuracy for Context2 choosing Target2: 51.00%
Balanced accuracy: 48.00%


[tensor([-2.1693, -2.6111, -2.9764, -6.9279, -3.7260, -1.2723, -2.1549],
        device='cuda:0'),
 tensor([ -2.0502,  -2.9551,  -1.2717, -12.8364,  -6.0774,  -3.3743,  -2.2576],
        device='cuda:0'),
 tensor([-1.6280, -3.9180, -0.3847, -8.0187, -2.4670], device='cuda:0'),
 tensor([-1.6227, -7.6699, -2.2626, -0.2149, -1.8072, -0.3040, -7.6421, -3.3018],
        device='cuda:0'),
 tensor([-5.2231, -0.3069, -1.9424, -6.9177, -1.0876], device='cuda:0'),
 tensor([-1.4210, -0.4645, -0.4206, -7.3121, -3.5311, -1.8979, -1.6361],
        device='cuda:0'),
 tensor([-1.9262, -7.7280, -2.9728, -0.4037, -1.2499, -0.6708, -6.5083, -1.2680],
        device='cuda:0'),
 tensor([-1.9385, -6.7047, -1.1797, -0.3695, -5.5078, -2.5416, -7.6151, -2.7257],
        device='cuda:0'),
 tensor([ -2.0466,  -3.9527,  -5.5849, -11.5113,  -1.8921], device='cuda:0'),
 tensor([ -1.3384,  -0.8082,  -6.4214,  -4.3963, -13.1611,  -1.2428],
        device='cuda:0'),
 tensor([ -2.6333,  -7.2639,  -5.6290, -11.0729,  -2

In [48]:
results[0]

tensor([-2.1693, -2.6111, -2.9764, -8.9879, -0.8838, -2.2774], device='cuda:0')

In [45]:
domains = ewok_df['Domain'].unique()

In [51]:
ewok_df

,Domain,ConceptA,ConceptB,ContextType,ContextDiff,TargetDiff,Context1,Context2,Target1,Target2
0,material-properties,fragile,sturdy,direct,antonym,concept swap,Jesse applied pressure to the balloon. It rema...,Jesse applied pressure to the balloon. It broke.,The balloon is sturdy.,The balloon is fragile.
1,material-properties,transparent,opaque,direct,negation,concept swap,Chao cannot see through the wheel.,Chao can see through the wheel.,The wheel is opaque.,The wheel is transparent.
2,material-properties,soft,hard,indirect,material,concept swap,The bin is made of jelly.,The bin is made of glass.,The bin is soft.,The bin is hard.
3,material-properties,soft,hard,indirect,material,concept swap,The balloon is made of terracotta.,The balloon is made of fabric.,The surface of the balloon is hard.,The surface of the balloon is soft.
4,material-properties,heavy,light,direct,antonym,concept swap,The balloon weighs a lot.,The balloon weighs little.,The balloon is heavy.,The balloon is light.
...,...,...,...,...,...,...,...,...,...,...
1095,material-dynamics,drape,stir,indirect,material,concept swap,Ali sees the propane.,Ali sees the twill.,Ali stirs it.,Ali drapes it.
1096,material-dynamics,drape,stir,direct,material,concept swap,Chao sees something that is liquid.,Chao sees something that is fabric.,Chao stirs it.,Chao drapes it.
1097,material-dynamics,squeeze,stir,indirect,material,concept swap,Mohammed sees the nitrogen.,Mohammed sees the play-doh.,Mohammed stirs it.,Mohammed squeezes it.
1098,material-dynamics,break,drip,indirect,material,concept swap,Chao sees the water.,Chao sees the feather.,It drips.,It breaks.


In [57]:
import torch

# --- 1. Setup Helper Functions ---
def sanitize(text):
    """
    Strips whitespace. 
    Crucial because 'Target' in CSV might look like ' Target' or 'Target ' 
    which messes up tokenization when concatenated.
    """
    return text.strip() if isinstance(text, str) else ""

def calculate_score(log_prob_tensor, normalize=True):
    if log_prob_tensor.numel() == 0: return -9999.0
    sum_log = log_prob_tensor.sum().item()
    return (sum_log / log_prob_tensor.numel()) if normalize else sum_log

# --- 2. Main Loop ---
# Toggle this based on what the specific benchmark paper says. 
# Most "zero-shot" benchmarks (like EWoK/BLiMP) use Length Normalization.
NORMALIZE = True 

for domain in domains:
    print(f"--- Domain: {domain} ---")
    
    # Filter and Copy to avoid SettingWithCopy warnings
    domain_df = ewok_df[ewok_df['Domain'] == domain].copy()
    if len(domain_df) == 0: continue

    # A. SANITIZE DATA (The likely fix for your 3% drift)
    # We strip all inputs so we control the spacing manually
    c1_raw = [sanitize(x) for x in domain_df['Context1'].tolist()]
    t1_raw = [sanitize(x) for x in domain_df['Target1'].tolist()]
    c2_raw = [sanitize(x) for x in domain_df['Context2'].tolist()]
    t2_raw = [sanitize(x) for x in domain_df['Target2'].tolist()]

    # B. Compute Log Probs
    # We pass 'batch_size=32' to run fast, but the logic inside handles one-by-one consistency
    res_c1_t1 = per_token_conditional_log_likelihood(model, tokenizer, c1_raw, t1_raw, batch_size=32)
    res_c1_t2 = per_token_conditional_log_likelihood(model, tokenizer, c1_raw, t2_raw, batch_size=32)
    res_c2_t2 = per_token_conditional_log_likelihood(model, tokenizer, c2_raw, t2_raw, batch_size=32)
    res_c2_t1 = per_token_conditional_log_likelihood(model, tokenizer, c2_raw, t1_raw, batch_size=32)

    # C. Calculate Scores
    scores_c1_t1 = [calculate_score(r, NORMALIZE) for r in res_c1_t1]
    scores_c1_t2 = [calculate_score(r, NORMALIZE) for r in res_c1_t2]
    scores_c2_t2 = [calculate_score(r, NORMALIZE) for r in res_c2_t2]
    scores_c2_t1 = [calculate_score(r, NORMALIZE) for r in res_c2_t1]

    # D. Compare (The Accuracy Check)
    # Question 1: Given Context 1, is Target 1 (Correct) > Target 2 (Wrong)?
    # We use >= to break ties in favor of the correct answer (optional, but common)
    correct_1 = sum([s1 > s2 for s1, s2 in zip(scores_c1_t1, scores_c1_t2)])
    
    # Question 2: Given Context 2, is Target 2 (Correct) > Target 1 (Wrong)?
    correct_2 = sum([s2 > s1 for s2, s1 in zip(scores_c2_t2, scores_c2_t1)])

    # E. Aggregate
    total_items = 2 * len(domain_df)
    total_correct = correct_1 + correct_2
    accuracy = total_correct / total_items
    
    print(f"Accuracy: {accuracy:.2%} ({total_correct}/{total_items})")

--- Domain: material-properties ---
Accuracy: 48.00% (96/200)
--- Domain: social-interactions ---
Accuracy: 52.00% (104/200)
--- Domain: social-relations ---
Accuracy: 50.00% (100/200)
--- Domain: spatial-relations ---
Accuracy: 47.00% (94/200)
--- Domain: quantitative-properties ---
Accuracy: 50.00% (100/200)
--- Domain: physical-dynamics ---
Accuracy: 48.50% (97/200)
--- Domain: agent-properties ---
Accuracy: 52.00% (104/200)
--- Domain: social-properties ---
Accuracy: 52.50% (105/200)
--- Domain: physical-relations ---
Accuracy: 49.00% (98/200)
--- Domain: physical-interactions ---
Accuracy: 50.00% (100/200)
--- Domain: material-dynamics ---
Accuracy: 50.50% (101/200)


In [ ]:
# import torch

# def get_sequence_log_probs(model, tokenizer, texts, device="cuda", batch_size=32):
#     """
#     Computes the sum of log probabilities for a list of texts (scalar score per text).
#     """
#     all_scores = []
    
#     for i in range(0, len(texts), batch_size):
#         batch_texts = texts[i : i + batch_size]
        
#         # Reuse your existing batch function
#         # It returns [Batch, Seq_Len] tensors of log probs
#         batch_log_probs, batch_attn_masks = per_token_log_likelihood(model, tokenizer, batch_texts, device)
        
#         # Sum each row to get the total log probability of the sequence
#         # We perform the sum on the GPU
#         batch_scores = batch_log_probs.sum(dim=1)
        
#         all_scores.append(batch_scores)
        
#     # Concatenate all batches into one 1D tensor
#     return torch.cat(all_scores)

# def compute_conditional_via_subtraction(model, tokenizer, contexts, targets, device="cuda", batch_size=32, normalize=True):
#     """
#     Computes P(Target|Context) = P(Context + Target) - P(Context).
#     """
#     # 1. Prepare texts
#     full_texts = [c + " " + t for c, t in zip(contexts, targets)]
    
#     # 2. Get scores for the FULL sequences (Context + Target)
#     full_scores = get_sequence_log_probs(model, tokenizer, full_texts, device, batch_size)
    
#     # 3. Get scores for the CONTEXT sequences only
#     context_scores = get_sequence_log_probs(model, tokenizer, contexts, device, batch_size)
    
#     # 4. Subtract to get conditional log likelihood of Target
#     # log P(T|C) = log P(T,C) - log P(C)
#     conditional_scores = full_scores - context_scores
    
#     # 5. Normalize by Target Length (Optional but recommended for benchmarks)
#     if normalize:
#         # We calculate target length by checking token counts
#         # (Approximate: Full_Len - Context_Len)
#         full_lens = torch.tensor([len(tokenizer.encode(t, add_special_tokens=False)) for t in full_texts], device=device)
#         context_lens = torch.tensor([len(tokenizer.encode(c, add_special_tokens=False)) for c in contexts], device=device)
#         target_lens = full_lens - context_lens
        
#         # Avoid division by zero
#         target_lens[target_lens == 0] = 1
#         conditional_scores = conditional_scores / target_lens

#     # Return as list of Python floats
#     return conditional_scores.tolist()

# # --- EVALUATION LOOP ---

# # Toggle Length Normalization (Most benchmarks use True)
# NORMALIZE = False

# for domain in domains:
#     print(f"--- Domain: {domain} ---")
#     domain_df = ewok_df[ewok_df['Domain'] == domain]
#     if len(domain_df) == 0: continue

#     # Extract lists
#     c1 = domain_df['Context1'].tolist()
#     t1 = domain_df['Target1'].tolist()
#     c2 = domain_df['Context2'].tolist()
#     t2 = domain_df['Target2'].tolist()

#     # Compute Conditional Scores using Subtraction Method
#     scores_c1_t1 = compute_conditional_via_subtraction(model, tokenizer, c1, t1, batch_size=32, normalize=NORMALIZE)
#     scores_c1_t2 = compute_conditional_via_subtraction(model, tokenizer, c1, t2, batch_size=32, normalize=NORMALIZE)
#     scores_c2_t2 = compute_conditional_via_subtraction(model, tokenizer, c2, t2, batch_size=32, normalize=NORMALIZE)
#     scores_c2_t1 = compute_conditional_via_subtraction(model, tokenizer, c2, t1, batch_size=32, normalize=NORMALIZE)

#     # Compare
#     correct_1 = sum([s1 > s2 for s1, s2 in zip(scores_c1_t1, scores_c1_t2)])
#     correct_2 = sum([s2 > s1 for s2, s1 in zip(scores_c2_t2, scores_c2_t1)])

#     total_items = 2 * len(domain_df)
#     accuracy = (correct_1 + correct_2) / total_items
    
#     print(f"Accuracy: {accuracy:.2%} ({correct_1 + correct_2}/{total_items})")

--- Domain: material-properties ---
Accuracy: 57.50% (115/200)
--- Domain: social-interactions ---
Accuracy: 44.00% (88/200)
--- Domain: social-relations ---
Accuracy: 50.50% (101/200)
--- Domain: spatial-relations ---
Accuracy: 47.50% (95/200)
--- Domain: quantitative-properties ---
Accuracy: 50.50% (101/200)
--- Domain: physical-dynamics ---
Accuracy: 55.50% (111/200)
--- Domain: agent-properties ---
Accuracy: 48.50% (97/200)
--- Domain: social-properties ---
Accuracy: 50.00% (100/200)
--- Domain: physical-relations ---
Accuracy: 51.50% (103/200)
--- Domain: physical-interactions ---
Accuracy: 51.50% (103/200)
--- Domain: material-dynamics ---
Accuracy: 42.00% (84/200)


In [61]:
import torch
import torch
from torch.nn import CrossEntropyLoss

def get_batch_log_prob_sums(model, tokenizer, texts, device="cuda", batch_size=32):
    """
    Computes the total log probability (scalar sum) for each text in the list.
    """
    model.eval()
    # We use right-padding for simple causal masking
    tokenizer.padding_side = "right" 
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    all_log_probs = []
    
    # We use CrossEntropyLoss to handle the math numerically stably
    # reduction='none' gives us loss per token
    loss_fct = CrossEntropyLoss(reduction='none', ignore_index=-100)

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        
        # 1. Tokenize
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True).to(device)
        input_ids = inputs.input_ids
        attn_mask = inputs.attention_mask
        
        # 2. Forward Pass
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attn_mask)
            logits = outputs.logits
        
        # 3. Shift for Causal LM (predict next token)
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = input_ids[..., 1:].contiguous()
        
        # 4. Calculate Loss
        # We need to mask the padding tokens in the loss calculation
        # The tokenizer mask usually handles 0s, but we set labels to -100 for safety
        shift_labels[attn_mask[..., 1:] == 0] = -100
        
        token_losses = loss_fct(shift_logits.transpose(1, 2), shift_labels)
        
        # 5. Sum Loss per row to get Sequence NLL
        # (CrossEntropy is positive NLL, so we negate to get LogProb)
        # Summing ignores -100 indices automatically
        sum_log_probs = -token_losses.sum(dim=1)
        
        all_log_probs.append(sum_log_probs)

    # Return a single 1D tensor of all scores
    return torch.cat(all_log_probs)

def per_token_conditional_log_likelihood(model, tokenizer, contexts, targets, device="cuda", batch_size=32):
    """
    Computes P(Target|Context) using the subtraction method:
    Score = P(Context + " " + Target) - P(Context)
    """
    # 1. Prepare Inputs
    #    Ensure we use the exact same spacing logic as your single-input success
    full_texts = [c + " " + t for c, t in zip(contexts, targets)]
    
    # 2. Compute P(Context + Target)
    #    This returns a tensor of shape [N]
    full_scores = get_batch_log_prob_sums(model, tokenizer, full_texts, device, batch_size)
    
    # 3. Compute P(Context)
    #    This returns a tensor of shape [N]
    context_scores = get_batch_log_prob_sums(model, tokenizer, contexts, device, batch_size)
    
    # 4. Subtract: log P(T|C) = log P(T,C) - log P(C)
    conditional_scores = full_scores - context_scores
    
    # 5. (Optional but likely needed for EWoK) Normalize by Target Length
    #    We estimate target length as (Full Tokens - Context Tokens)
    #    This avoids the issue where shorter targets win automatically.
    full_lens = torch.tensor([len(tokenizer.encode(t, add_special_tokens=False)) for t in full_texts], device=device)
    context_lens = torch.tensor([len(tokenizer.encode(c, add_special_tokens=False)) for c in contexts], device=device)
    target_lens = full_lens - context_lens
    target_lens[target_lens == 0] = 1 # Safety
    
    normalized_scores = conditional_scores 
    
    # Return as list of floats to match your evaluation loop format
    return normalized_scores.tolist()
# --- EVALUATION LOOP ---
for domain in domains:
    print(f"--- Domain: {domain} ---")
    domain_df = ewok_df[ewok_df['Domain'] == domain]
    if len(domain_df) == 0: continue

    c1 = domain_df['Context1'].tolist()
    t1 = domain_df['Target1'].tolist()
    c2 = domain_df['Context2'].tolist()
    t2 = domain_df['Target2'].tolist()

    # These return LISTS of float scores
    scores_c1_t1 = per_token_conditional_log_likelihood(model, tokenizer, c1, t1)
    scores_c1_t2 = per_token_conditional_log_likelihood(model, tokenizer, c1, t2)
    scores_c2_t2 = per_token_conditional_log_likelihood(model, tokenizer, c2, t2)
    scores_c2_t1 = per_token_conditional_log_likelihood(model, tokenizer, c2, t1)

    # Compare
    correct_1 = sum([s1 > s2 for s1, s2 in zip(scores_c1_t1, scores_c1_t2)])
    correct_2 = sum([s2 > s1 for s2, s1 in zip(scores_c2_t2, scores_c2_t1)])

    total = 2 * len(domain_df)
    acc = (correct_1 + correct_2) / total
    print(f"Accuracy: {acc:.2%} ({correct_1 + correct_2}/{total})")

--- Domain: material-properties ---
Accuracy: 48.00% (96/200)
--- Domain: social-interactions ---
Accuracy: 50.50% (101/200)
--- Domain: social-relations ---
Accuracy: 51.00% (102/200)
--- Domain: spatial-relations ---
Accuracy: 46.50% (93/200)
--- Domain: quantitative-properties ---
Accuracy: 52.00% (104/200)
--- Domain: physical-dynamics ---
Accuracy: 50.00% (100/200)
--- Domain: agent-properties ---
Accuracy: 52.50% (105/200)
--- Domain: social-properties ---
Accuracy: 52.00% (104/200)
--- Domain: physical-relations ---
Accuracy: 49.50% (99/200)
--- Domain: physical-interactions ---
Accuracy: 50.50% (101/200)
--- Domain: material-dynamics ---
Accuracy: 49.00% (98/200)


In [9]:
results

tensor([ -8.3853,  -5.1996, -24.8474])